# 07 · Agregación de las simulaciones y tablas de resultados

Lee las réplicas Montecarlo de los dos escenarios (notebook 06), las resume con la mediana y el
intervalo 2,5-97,5 % entre réplicas, y prepara los insumos observados con los que se empalman
las series proyectadas. Todo lo que dibuja el notebook 08 sale de las tablas que se escriben acá.

**Resultados por escenario** (`resultados/tablas/resultados/`):

| Archivo | Uso en la tesis |
|---|---|
| `tgf_proyectada.csv` | Figura *TGF proyectada, Escenario Base y Alternativo* |
| `natalidad_mortalidad_{esc}.csv`, `tbn_observada.csv` | Figura *Tasa bruta de natalidad y de mortalidad* |
| `tramos_funcionales_{esc}.csv` | Figura *Composición de la población por grupos de edad* |
| `piramides.csv` | Figura *Estructura de la población por edad y sexo* (2010, 2025, 2050 y 2075) |
| `matricula_{esc}.csv` | Figura *Matrícula proyectada por nivel del sistema educativo* |
| `composicion_educativa_25_65_{esc}.csv` | Figura *Composición educativa de la población de 25 a 65 años* |
| `nacimientos_clima.csv`, `nacimientos_clima_2010.csv` | Figura *Composición de los nacimientos por clima educativo* |
| `fecundidad_generacional_{esc}.csv` | Tabla *Completed fertility por generación* |

**Descendencia final por generación.** Para las generaciones observadas en la EPH se toma la
última observación de cada cohorte y se la lleva, con el modelo de la cantidad total de hijos
(notebook 04), a la edad de mayor acumulación de hijos convivientes (≈ 42 años). Para las
generaciones simuladas se usa la paridad acumulada de cada cohorte a los 45 años (o a la última
edad que alcanza en 2075), completada con el mismo modelo. Las filas con más de 30 % de valor
extrapolado no se reportan.

**Entradas:** `resultados/intermedios/simulaciones/{base,rebote}/` (se puede apuntar a otra
carpeta con la variable de entorno `DIR_SIMULACIONES`), `poblacion_base.csv`, modelos del
notebook 04, `datos/eph/` (EPH 2016-2024 y EAHU 2010).

**Tiempo aproximado:** 10-15 minutos con 1.000 + 1.000 réplicas.

In [1]:
import glob
import os
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
DIR_SIM = Path(os.environ.get("DIR_SIMULACIONES", "resultados/intermedios/simulaciones"))
DIR_TAB = Path("resultados/tablas/resultados")
DIR_TAB.mkdir(parents=True, exist_ok=True)
RUTA_POB = "resultados/intermedios/poblacion_base/poblacion_base.csv"
DIR_MOD = Path("resultados/intermedios/modelos")
ESCENARIOS = ["base", "rebote"]

ANIO_INI_PANEL, ANIO_FIN_PANEL = 2016, 2024
N_ESTRATOS_STATUS = 3
STATUS_LEVELS = list(range(1, N_ESTRATOS_STATUS + 1))
SEMILLA_IMPUTACION, MIN_DONANTES_IMPUTACION = 20260905, 10
EDAD_FIN_FERTIL = 45
EDAD_PROXY_NACIMIENTOS = 3
ANIOS_PIRAMIDE = (2050, 2075)
TRAMOS_ESCOLARES = [("Inicial (3-5)", 3, 5), ("Primaria (6-11)", 6, 11),
                    ("Secundaria (12-17)", 12, 17), ("Superior (18-24)", 18, 24)]
TRAMOS_FUNCIONALES = [("Menores de 18", lambda a, s: a < 18),
                      ("En edad de sistema educativo (<25)", lambda a, s: a < 25),
                      ("Población en edad activa, 15-64", lambda a, s: (a >= 15) & (a <= 64)),
                      ("Mujeres en edad fértil, 15-49", lambda a, s: (s == 1) & (a >= 15) & (a <= 49)),
                      ("Personas Mayores, 65 y más", lambda a, s: a >= 65)]
CLIMA3 = {0: "Bajo", 1: "Bajo", 2: "Medio", 3: "Alto", 4: "Alto"}
PROVINCIA_REGION = {1: 1, 2: 1, 3: 2, 4: 2, 5: 2, 6: 2, 7: 2, 8: 2, 9: 3, 10: 3, 11: 3, 12: 3,
                    13: 4, 14: 4, 15: 4, 16: 5, 17: 5, 18: 5, 19: 5, 20: 6, 21: 6, 22: 6, 23: 6, 24: 6}
aglomerado_mapeo = {32: 1, 33: 2, 2: 2, 3: 2, 34: 2, 38: 2, 40: 2, 18: 3, 61: 3, 19: 4, 48: 4,
                    22: 5, 41: 5, 23: 6, 55: 6, 25: 7, 50: 7, 29: 8, 62: 8, 7: 9, 52: 9,
                    8: 10, 44: 10, 12: 11, 43: 11, 15: 12, 47: 12, 10: 13, 51: 13, 26: 14, 57: 14,
                    27: 15, 56: 15, 4: 16, 5: 16, 60: 16, 6: 17, 14: 17, 46: 17, 13: 18, 36: 18,
                    42: 18, 30: 19, 49: 19, 9: 20, 91: 20, 45: 20, 17: 21, 53: 21, 20: 22, 58: 22,
                    31: 23, 93: 24, 54: 24}
NIVEL_EDU7 = {0: "0 · Sin instrucción", 1: "1 · Primaria incompleta", 2: "2 · Primaria completa",
              3: "3 · Secundaria incompleta", 4: "4 · Secundaria completa", 5: "5 · Superior incompleta",
              6: "6 · Superior completa"}
ORDEN_EDU7 = list(NIVEL_EDU7.values())


def resumen_por_anio(df, col, por="year"):
    return (df.groupby(por)[col].agg(mediana="median", p025=lambda s: s.quantile(0.025),
                                     p975=lambda s: s.quantile(0.975), n_k="size").reset_index())

## Lectura de las réplicas

In [2]:
def tramo_escolar(edad):
    for etiqueta, lo, hi in TRAMOS_ESCOLARES:
        if lo <= edad <= hi:
            return etiqueta
    return None


MAPA_TRAMO = {a: tramo_escolar(a) for a in range(0, 130)}


def leer(ruta, grupos, columnas):
    return pd.read_parquet(ruta, columns=columnas, filters=[("grupo", "in", grupos)])


def partir(categoria, n):
    return categoria.str.split("_", expand=True).iloc[:, :n].astype(float).astype(int)


def leer_escenario(escenario):
    d = DIR_SIM / escenario
    ks = sorted(int(Path(f).stem.split("_k")[1]) for f in glob.glob(str(d / "educ_k*.parquet")))
    acum = {c: [] for c in ("tgf", "total", "tramos", "piramide", "matricula", "edu2565", "clima",
                            "cohortes", "edad_educ")}
    for i, k in enumerate(ks, 1):
        fert = leer(d / f"fert_k{k:04d}.parquet", ["tgf_sintetica", "cohorte_birth_year_x_parity", "age_x_education"],
                    ["k", "year", "grupo", "categoria", "women_n", "births_n", "children_n", "tgf_sintetica"])
        acum["tgf"].append(fert.loc[fert.grupo == "tgf_sintetica", ["k", "year", "tgf_sintetica", "births_n"]])

        c = fert[fert.grupo == "cohorte_birth_year_x_parity"].copy()
        c["birth_year"] = partir(c["categoria"], 1)[0].values
        c["edad"] = c["year"] - c["birth_year"]
        agg = c.groupby(["k", "year", "birth_year", "edad"], as_index=False).agg(
            women_n=("women_n", "sum"), children_n=("children_n", "sum"))
        agg["paridad_media"] = agg["children_n"] / agg["women_n"].replace(0, np.nan)
        agg = agg[agg["edad"].between(15, 50)]
        agg["_pref"] = np.where(agg["edad"] <= EDAD_FIN_FERTIL, agg["edad"], -1)
        acum["cohortes"].append(agg.loc[agg.groupby(["k", "birth_year"])["_pref"].idxmax()]
                                .dropna(subset=["paridad_media"]).drop(columns="_pref"))

        ae = fert[fert.grupo == "age_x_education"]
        p = partir(ae["categoria"], 2)
        acum["edad_educ"].append(pd.DataFrame({"year": ae["year"].values.astype(np.int16),
                                               "age": p[0].values.astype(np.int16),
                                               "education": p[1].values.astype(np.int8),
                                               "women_n": ae["women_n"].values.astype(float)}))

        pop = leer(d / f"pop_k{k:04d}.parquet", ["total", "sex_x_age"], ["k", "year", "grupo", "categoria", "n"])
        tot = pop.loc[pop.grupo == "total", ["k", "year", "n"]].rename(columns={"n": "tot"})
        acum["total"].append(tot)
        sa = pop[pop.grupo == "sex_x_age"].copy()
        p = partir(sa["categoria"], 2)
        sa["sex"], sa["age"] = p[0].values, p[1].values
        for etiqueta, fn in TRAMOS_FUNCIONALES:
            m = fn(sa["age"].to_numpy(), sa["sex"].to_numpy())
            t = sa[m].groupby(["k", "year"], as_index=False)["n"].sum().merge(tot, on=["k", "year"], how="right").fillna(0)
            acum["tramos"].append(t.assign(v=t["n"] / t["tot"] * 100, tramo=etiqueta)[["k", "year", "tramo", "v"]])
        acum["piramide"].append(sa.loc[sa.year.isin(ANIOS_PIRAMIDE), ["k", "year", "sex", "age", "n"]])

        educ = leer(d / f"educ_k{k:04d}.parquet",
                    ["age_x_in_system", "age_x_education", "age_x_in_system_x_clima_educativo"],
                    ["k", "year", "grupo", "categoria", "n"])
        ais = educ[educ.grupo == "age_x_in_system"].copy()
        p = partir(ais["categoria"], 2)
        ais["age"], ais["in_system"] = p[0].values, p[1].values
        ais["tramo"] = ais["age"].map(MAPA_TRAMO)
        acum["matricula"].append(ais[(ais.in_system == 1) & ais.tramo.notna()]
                                 .groupby(["k", "year", "tramo"], as_index=False)["n"].sum())
        e = educ[educ.grupo == "age_x_education"].copy()
        p = partir(e["categoria"], 2)
        e["age"], e["edu"] = p[0].values, p[1].values
        acum["edu2565"].append(e[(e.age >= 25) & (e.age <= 65)].groupby(["k", "year", "edu"], as_index=False)["n"].sum())
        cl = educ[educ.grupo == "age_x_in_system_x_clima_educativo"].copy()
        p = partir(cl["categoria"], 3)
        cl = cl[p[0].values == EDAD_PROXY_NACIMIENTOS].copy()
        cl["clima3"] = p.loc[cl.index, 2].map(CLIMA3).values
        acum["clima"].append(cl.groupby(["k", "year", "clima3"], as_index=False)["n"].sum())
        if i % 200 == 0:
            print(f"  [{escenario}] {i}/{len(ks)} réplicas leídas", flush=True)
    print(f"[{escenario}] {len(ks)} réplicas")
    return {c: pd.concat(v, ignore_index=True) for c, v in acum.items()}


SIM = {e: leer_escenario(e) for e in ESCENARIOS}

  [base] 200/1000 réplicas leídas


  [base] 400/1000 réplicas leídas


  [base] 600/1000 réplicas leídas


  [base] 800/1000 réplicas leídas


  [base] 1000/1000 réplicas leídas


...

  [rebote] 400/1000 réplicas leídas


  [rebote] 600/1000 réplicas leídas


  [rebote] 800/1000 réplicas leídas


  [rebote] 1000/1000 réplicas leídas


[rebote] 1000 réplicas


## TGF, natalidad y mortalidad, grupos de edad, matrícula y composición educativa

In [3]:
r = {e: resumen_por_anio(SIM[e]["tgf"], "tgf_sintetica") for e in ESCENARIOS}
tgf = r["base"][["year"]].copy()
for e in ESCENARIOS:
    tgf[[f"tgf_{e}", f"tgf_{e}_p025", f"tgf_{e}_p975"]] = r[e][["mediana", "p025", "p975"]].values
tgf.to_csv(DIR_TAB / "tgf_proyectada.csv", index=False)

for e in ESCENARIOS:
    vit = (SIM[e]["total"].rename(columns={"tot": "pob"})
           .merge(SIM[e]["tgf"][["k", "year", "births_n"]], on=["k", "year"], how="inner").sort_values(["k", "year"]))
    vit["pob_prev"] = vit.groupby("k")["pob"].shift(1)
    vit["defunciones"] = vit["pob_prev"] - vit["pob"] + vit["births_n"]
    vit["tbn"] = vit["births_n"] / vit["pob"] * 1000
    vit["tbm"] = vit["defunciones"] / vit["pob"] * 1000
    out = resumen_por_anio(vit, "tbn").add_prefix("tbn_").rename(columns={"tbn_year": "year"})
    out = out.merge(resumen_por_anio(vit.dropna(subset=["tbm"]), "tbm").add_prefix("tbm_")
                    .rename(columns={"tbm_year": "year"}), on="year", how="left")
    out = out.merge(resumen_por_anio(vit, "births_n").add_prefix("nacimientos_").rename(
        columns={"nacimientos_year": "year"}), on="year", how="left")
    out.to_csv(DIR_TAB / f"natalidad_mortalidad_{e}.csv", index=False)

    tr = pd.concat([resumen_por_anio(SIM[e]["tramos"][SIM[e]["tramos"].tramo == t], "v").assign(tramo=t)
                    for t, _ in TRAMOS_FUNCIONALES], ignore_index=True)
    tr.to_csv(DIR_TAB / f"tramos_funcionales_{e}.csv", index=False)

    mt = pd.concat([resumen_por_anio(SIM[e]["matricula"][SIM[e]["matricula"].tramo == t], "n").assign(tramo=t)
                    for t, _, _ in TRAMOS_ESCOLARES], ignore_index=True)
    mt.to_csv(DIR_TAB / f"matricula_{e}.csv", index=False)

tgf.round(3).iloc[[0, 17, 49]]

,year,tgf_base,tgf_base_p025,tgf_base_p975,tgf_rebote,tgf_rebote_p025,tgf_rebote_p975
0,2026,1.181,1.177,1.185,1.212,1.208,1.216
17,2043,1.033,1.015,1.051,1.052,1.031,1.070
49,2075,1.003,0.980,1.023,1.249,1.224,1.276


In [4]:
def historico_eph_niveles(edad_min, edad_max):
    filas = []
    for a in range(ANIO_INI_PANEL, ANIO_FIN_PANEL + 1):
        d = pd.read_csv(f"datos/eph/personas_tot.urb_3T_{a}.txt", sep=";", low_memory=False)
        d.columns = d.columns.str.lower()
        age = pd.to_numeric(d["ch06"], errors="coerce").fillna(-1).astype(int)
        edu = pd.to_numeric(d["nivel_ed"], errors="coerce").fillna(7).astype(int).replace(7, 0)
        pond = pd.to_numeric(d["pondera"], errors="coerce").fillna(0.0)
        m = (age >= edad_min) & (age <= edad_max)
        w = pd.DataFrame({"nivel": edu[m].map(NIVEL_EDU7), "pondera": pond[m]}).groupby("nivel")["pondera"].sum()
        w = w / w.sum() * 100
        filas.append({"year": a, **{n: float(w.get(n, 0.0)) for n in ORDEN_EDU7}})
    return pd.DataFrame(filas)


poblacion_base = pd.read_csv(RUTA_POB)
obs_2565 = historico_eph_niveles(25, 65)
p = poblacion_base[poblacion_base["age"].between(25, 65)]
w = p.assign(nivel=p["education"].map(NIVEL_EDU7)).groupby("nivel")["n"].sum()
obs_2565 = pd.concat([obs_2565, pd.DataFrame([{"year": 2025, **(w / w.sum() * 100).reindex(ORDEN_EDU7).fillna(0.0)}])],
                     ignore_index=True)
for e in ESCENARIOS:
    d = SIM[e]["edu2565"].copy()
    d["nivel"] = d["edu"].map(NIVEL_EDU7)
    d["share"] = d["n"] / d.groupby(["k", "year"])["n"].transform("sum") * 100
    proy = d.groupby(["year", "nivel"])["share"].median().unstack().reindex(columns=ORDEN_EDU7).fillna(0.0)
    proy = proy.div(proy.sum(axis=1), axis=0) * 100
    piv = pd.concat([obs_2565.set_index("year")[ORDEN_EDU7], proy]).sort_index()
    piv = piv.div(piv.sum(axis=1), axis=0) * 100
    piv.reset_index(names="year").assign(tramo=lambda t: np.where(t["year"] <= 2025, "observado", "proyectado")) \
       .to_csv(DIR_TAB / f"composicion_educativa_25_65_{e}.csv", index=False)
pd.read_csv(DIR_TAB / "composicion_educativa_25_65_base.csv").set_index("year").loc[[2016, 2031, 2050, 2075]].round(1)

,0 · Sin instrucción,1 · Primaria incompleta,2 · Primaria completa,3 · Secundaria incompleta,4 · Secundaria completa,5 · Superior incompleta,6 · Superior completa,tramo
year,,,,,,,,
2016,0.7,6.1,20.9,16.3,26.0,11.0,19.0,observado
2031,0.4,3.3,10.1,16.5,27.6,18.0,24.1,proyectado
2050,0.5,3.3,7.5,26.0,24.4,17.9,20.4,proyectado
2075,2.2,4.5,12.5,31.8,20.9,12.8,15.4,proyectado


## Pirámides y origen social de los nacimientos (observado 2010 en la EAHU)

In [5]:
def get_ed_years_simple(df):
    nivel = pd.to_numeric(df["ch12"], errors="coerce")
    finalizado = pd.to_numeric(df["ch13"], errors="coerce")
    aprobado = pd.to_numeric(df["ch14"], errors="coerce")
    educacion = pd.to_numeric(df["nivel_ed"], errors="coerce").fillna(7).replace(7, 0)
    primaria, egb, secundaria, terciario, universidad, posgrado = 6, 9, 12, 14, 16, 18
    aprobado_valido = aprobado.notna() & ~aprobado.isin([98, 99])
    aprobado_safe = aprobado.fillna(0)
    fin1 = finalizado == 1
    vals = [pd.Series(0.0, index=df.index),
            np.select([fin1, aprobado_valido, educacion == 2],
                      [primaria, np.minimum(aprobado_safe, primaria - 0.5), primaria], default=primaria / 2),
            np.select([fin1, aprobado_valido, educacion == 2],
                      [egb, np.minimum(aprobado_safe, egb - 0.5), primaria], default=egb / 2),
            np.select([fin1, aprobado_valido, educacion == 4],
                      [secundaria, np.minimum(primaria + aprobado_safe, secundaria - 0.5), secundaria], default=primaria + 3),
            np.select([fin1, aprobado_valido, educacion == 4],
                      [secundaria, np.minimum(egb + aprobado_safe, secundaria - 0.5), secundaria], default=egb + 1.5),
            np.select([fin1, aprobado_valido, educacion == 6],
                      [terciario, np.minimum(secundaria + aprobado_safe, terciario - 0.5), terciario], default=secundaria + 1),
            np.select([fin1, aprobado_valido, educacion == 6],
                      [universidad, np.minimum(secundaria + aprobado_safe, universidad - 0.5), universidad],
                      default=secundaria + 2.5),
            np.select([fin1, aprobado_valido], [posgrado, np.minimum(universidad + aprobado_safe, posgrado - 0.5)],
                      default=universidad + 1),
            np.select([educacion == i for i in range(1, 7)],
                      [primaria / 2, primaria, primaria + 3, secundaria, secundaria + 2, universidad], default=0.0)]
    condiciones = [nivel == i for i in range(1, 9)] + [nivel.isin([9, 99]) | nivel.isna()]
    return np.select(condiciones, vals, default=0.0)


eahu = pd.read_stata("datos/eph/Individual_EAHU_10.dta", convert_categoricals=False)
eahu.columns = eahu.columns.str.lower()
eahu["age"] = pd.to_numeric(eahu["ch06"], errors="coerce").fillna(-1).astype(int)
eahu["pondera"] = pd.to_numeric(eahu["pondera"], errors="coerce").fillna(0.0)

piramides = [eahu.assign(age=eahu["age"].clip(lower=0), sex=eahu["ch04"].map({2: 1, 1: 0}))
             .query("age <= 100 and sex == sex").groupby(["age", "sex"], as_index=False)["pondera"].sum()
             .rename(columns={"pondera": "n"}).assign(panel="2010 · observado (EAHU)")]
p25 = poblacion_base.groupby(["age", "sex"], as_index=False)["n"].sum()
piramides.append(p25[p25["age"] <= 100].assign(panel="2025 · población base"))
pir = SIM["base"]["piramide"]
for a in ANIOS_PIRAMIDE:
    piramides.append(pir[pir.year == a].groupby(["sex", "age"])["n"].median().reset_index()
                     .assign(panel=f"{a} · proyectado"))
pd.concat(piramides, ignore_index=True)[["panel", "sex", "age", "n"]].to_csv(DIR_TAB / "piramides.csv", index=False)

eahu["hogar_id"] = eahu["codusu"].astype(str) + "_" + eahu["nro_hogar"].astype(str)
eahu["ed_years"] = get_ed_years_simple(eahu)
eahu = eahu.merge(eahu[eahu["age"] >= 18].groupby("hogar_id")["ed_years"].mean().rename("average_years"),
                  on="hogar_id", how="left")
eahu["clima3"] = (pd.cut(eahu["average_years"], bins=[-np.inf, 7, 11, 14, 16, np.inf], labels=[0, 1, 2, 3, 4],
                         right=False).astype("float").fillna(0).astype(int).map(CLIMA3))
w2010 = eahu[eahu["age"] == 0].groupby("clima3")["pondera"].sum()
(w2010 / w2010.sum() * 100).reindex(["Bajo", "Medio", "Alto"]).rename("share_nacimientos_pct") \
    .reset_index().to_csv(DIR_TAB / "nacimientos_clima_2010.csv", index=False)

series = []
for e in ESCENARIOS:
    s = SIM[e]["clima"].copy()
    s["share"] = s["n"] / s.groupby(["k", "year"])["n"].transform("sum") * 100
    s["anio_nacimiento"] = s["year"] - EDAD_PROXY_NACIMIENTOS
    series.append(resumen_por_anio(s, "share", por=["anio_nacimiento", "clima3"]).assign(escenario=e))
nacimientos_clima = pd.concat(series, ignore_index=True)
nacimientos_clima.to_csv(DIR_TAB / "nacimientos_clima.csv", index=False)
nacimientos_clima[nacimientos_clima.anio_nacimiento.isin([2050, 2070])].pivot_table(
    index=["escenario", "anio_nacimiento"], columns="clima3", values="mediana").round(1)

clima3                     Alto  Bajo  Medio
escenario anio_nacimiento                   
base      2050             16.5  52.5   31.0
          2070             13.9  56.1   30.0
rebote    2050             16.2  52.8   31.0
          2070             13.4  56.7   29.9

## Panel EPH 2016-2024 y tasa bruta de natalidad observada

In [6]:
def get_ingreso_adulto(df, col_hogar='hogar_id', col_edad='ch06'):
    itf = pd.to_numeric(df['itf'].astype(str).str.replace(',', '.', regex=False), errors='coerce')
    adultos = df.groupby(col_hogar)[col_edad].transform(lambda s: (s >= 18).sum()).clip(lower=1)
    return itf / adultos


def get_weighted_quantiles(df, value_col, weight_col, k=N_ESTRATOS_STATUS):
    valid = df[value_col].notna() & df[weight_col].notna() & (df[weight_col] > 0)
    df_valid = df[valid].sort_values(value_col, kind='mergesort')
    cum = df_valid[weight_col].cumsum()
    q = pd.cut(cum / cum.iloc[-1], bins=np.linspace(0, 1, k + 1), labels=range(1, k + 1), include_lowest=True)
    return q.astype('Int64').reindex(df.index)


def imputar_status(df, seed=SEMILLA_IMPUTACION, k=N_ESTRATOS_STATUS, min_donantes=MIN_DONANTES_IMPUTACION):
    rng = np.random.default_rng(seed)
    st = pd.to_numeric(df['status'], errors='coerce').to_numpy(dtype=float)
    w = pd.to_numeric(df['pondih'], errors='coerce').fillna(0.0).to_numpy(dtype=float)
    for keys in [['education', 'sex', 'clima_educativo', 'provincia'], ['education', 'sex'], None]:
        if not np.isnan(st).any():
            break
        grupos = ({None: np.arange(len(df))} if keys is None
                  else df.groupby(keys, dropna=False, observed=True).indices)
        for pos in grupos.values():
            pos = np.asarray(pos)
            falta = np.isnan(st[pos])
            if not falta.any():
                continue
            don = pos[~falta]
            if len(don) < min_donantes:
                continue
            pw = w[don]
            if not np.isfinite(pw).all() or pw.sum() <= 0:
                pw = np.ones(len(don))
            st[pos[falta]] = rng.choice(st[don], size=int(falta.sum()), replace=True, p=pw / pw.sum())
    resto = np.isnan(st)
    if resto.any():
        st[resto] = rng.integers(1, k + 1, size=int(resto.sum()))
    return pd.Series(st, index=df.index).round().astype(int)


def get_ed_years(df):
    nivel, finalizado, aprobado, educacion = df['ch12'], df['ch13'], df['ch14'], df['education']
    primaria, egb, secundaria, terciario, universidad, posgrado = 6, 9, 12, 14, 16, 18
    aprobado_valido = aprobado.notna() & ~aprobado.isin([98, 99])
    aprobado_safe = aprobado.fillna(0)
    fin1 = finalizado == 1
    vals = [pd.Series(0.0, index=df.index),
            np.select([fin1, aprobado_valido, educacion == 2],
                      [primaria, np.minimum(aprobado_safe, primaria - 0.5), primaria], default=primaria / 2),
            np.select([fin1, aprobado_valido, educacion == 2],
                      [egb, np.minimum(aprobado_safe, egb - 0.5), primaria], default=egb / 2),
            np.select([fin1, aprobado_valido, educacion == 4],
                      [secundaria, np.minimum(primaria + aprobado_safe, secundaria - 0.5), secundaria], default=primaria + 3),
            np.select([fin1, aprobado_valido, educacion == 4],
                      [secundaria, np.minimum(egb + aprobado_safe, secundaria - 0.5), secundaria], default=egb + 1.5),
            np.select([fin1, aprobado_valido, educacion == 6],
                      [terciario, np.minimum(secundaria + aprobado_safe, terciario - 0.5), terciario], default=secundaria + 1),
            np.select([fin1, aprobado_valido, educacion == 6],
                      [universidad, np.minimum(secundaria + aprobado_safe, universidad - 0.5), universidad],
                      default=secundaria + 2.5),
            np.select([fin1, aprobado_valido], [posgrado, np.minimum(universidad + aprobado_safe, posgrado - 0.5)],
                      default=universidad + 1),
            np.select([educacion == i for i in range(1, 7)],
                      [primaria / 2, primaria, primaria + 3, secundaria, secundaria + 2, universidad], default=0.0)]
    condiciones = [nivel == i for i in range(1, 9)] + [nivel.isin([9, 99]) | nivel.isna()]
    return np.select(condiciones, vals, default=0.0)


def asignar_nietos_a_hijas(df, edad_col):
    idx = df.index
    vacio_n, vacio_age, vacio_m = pd.Series(0.0, index=idx), pd.Series(99.0, index=idx), pd.Series(0, index=idx)
    nietos = (df.loc[df["ch03"] == 5, ["hogar_id", edad_col]].rename(columns={edad_col: "edad_nieto"})
              .reset_index().rename(columns={"index": "idx_nieto"}))
    hijas = (df.loc[(df["ch03"] == 3) & (df["sex"] == 1) & (df[edad_col].between(15, 50)), ["hogar_id", edad_col]]
             .rename(columns={edad_col: "edad_hija"}).reset_index().rename(columns={"index": "idx_hija"}))
    if nietos.empty or hijas.empty:
        return vacio_n, vacio_age, vacio_m
    pares = nietos.merge(hijas, on="hogar_id", how="inner")
    pares = pares[pares["edad_hija"] >= pares["edad_nieto"] + 15]
    if pares.empty:
        return vacio_n, vacio_age, vacio_m
    pares = pares.sort_values(["idx_nieto", "edad_hija", "idx_hija"], ascending=[True, False, True])
    adj = pares.drop_duplicates(subset="idx_nieto", keep="first")
    ma = pd.Series(idx.map(adj.groupby("idx_hija")["edad_nieto"].min()), index=idx).astype(float).fillna(99.0)
    return (pd.Series(idx.map(adj.groupby("idx_hija").size()), index=idx).astype(float).fillna(0.0), ma,
            (ma == 0).astype(int))


partes, poblacion_hist = [], []
for y in range(ANIO_INI_PANEL, ANIO_FIN_PANEL + 1):
    df = pd.read_csv(f"datos/eph/personas_tot.urb_3T_{y}.txt", sep=";", low_memory=False)
    df.columns = df.columns.str.lower()
    poblacion_hist.append({"year": y, "poblacion": float(pd.to_numeric(df["pondera"], errors="coerce")
                                                         .fillna(0.0).clip(lower=0).sum())})
    df["ch06"] = df["ch06"].astype(int).replace(-1, 0)
    df["year"] = df["ano4"].astype(int)
    df["education"] = df["nivel_ed"].astype(int).replace(7, 0)
    df["sex"] = df["ch04"].map({2: 1, 1: 0})
    df["provincia"] = df["aglomerado"].map(aglomerado_mapeo).fillna(0).astype(int)
    df["hogar_id"] = df["codusu"] + "_" + df["nro_hogar"].astype(str) + "_" + df["ano4"].astype(str)
    df["ed_years"] = get_ed_years(df)
    df = df.merge(df[df["ch06"] >= 18].groupby("hogar_id")["ed_years"].mean().rename("average_years"),
                  on="hogar_id", how="left")
    df["clima_educativo"] = (pd.cut(df["average_years"], bins=[-np.inf, 7, 11, 14, 16, np.inf], labels=[0, 1, 2, 3, 4],
                                    right=False).astype("float").fillna(0).astype(int))
    df["ipcf"] = pd.to_numeric(df["ipcf"], errors="coerce")
    df["ingreso_adulto"] = get_ingreso_adulto(df, col_edad="ch06")
    df["status"] = get_weighted_quantiles(df, "ingreso_adulto", "pondih")
    df["status"] = imputar_status(df)
    df["work"] = (df["estado"] == 1).astype(int)
    df["couple"] = df["ch07"].isin([1, 2]).astype(int)
    df["insurance"] = pd.to_numeric(df["ch08"], errors="coerce").isin([1, 2, 12, 13, 23, 123]).astype(int)
    df["formality"] = (pd.to_numeric(df["pp07h"], errors="coerce") == 1).astype(int)
    jefes = df[df.ch03 == 1][["hogar_id", "work"]].rename(columns={"work": "j_work"})
    con = df[df.ch03 == 2][["hogar_id", "work"]].rename(columns={"work": "c_work"})
    df = df.merge(pd.merge(jefes, con, on="hogar_id", how="outer"), on="hogar_id", how="left")
    df = df.merge(df[df.ch03 == 3].groupby("hogar_id").agg(children_jefe=("ch06", "count"),
                                                           min_child_age_jefe=("ch06", "min")).reset_index(),
                  on="hogar_id", how="left")
    df["children_jefe"] = df["children_jefe"].fillna(0)
    na, ma, _ = asignar_nietos_a_hijas(df, edad_col="ch06")
    m = df[(df.sex == 1) & (df.ch06.between(15, 50))].copy()
    cj, ch = m.ch03.isin([1, 2]), (m.ch03 == 3)
    m["target_children"] = np.where(cj, m.children_jefe, np.where(ch, na.loc[m.index], 0))
    m["youngest_age"] = pd.Series(np.where(cj, m.min_child_age_jefe, np.where(ch, ma.loc[m.index], 99)),
                                  index=m.index).fillna(99)
    m["target_motherhood"] = (m.youngest_age == 0).astype(int)
    m["s_work"] = pd.Series(np.where(m.ch03 == 1, m.c_work, np.where(m.ch03 == 2, m.j_work, np.nan)),
                            index=m.index).fillna(0)
    m["hh_work"] = np.maximum(m.work, m.s_work)
    m = m.rename(columns={"ch06": "age"})
    m["region"] = m["provincia"].map(PROVINCIA_REGION)
    m["birth_year"] = m["year"] - m["age"]
    partes.append(m[["year", "age", "birth_year", "region", "education", "status", "pondera", "hh_work", "couple",
                     "insurance", "formality", "target_children", "target_motherhood"]])
panel_eph = pd.concat(partes, ignore_index=True)
for s in STATUS_LEVELS:
    panel_eph[f"status{s}"] = (panel_eph["status"] == s).astype(int)

factor = pd.read_csv("resultados/intermedios/eph_corregida/factores_correccion_edad0.csv").set_index("year")["factor_f"]
tbn_obs = (panel_eph.assign(nac=panel_eph["pondera"] * panel_eph["target_motherhood"]).groupby("year")["nac"].sum()
           .reset_index().merge(pd.DataFrame(poblacion_hist), on="year"))
tbn_obs["tbn"] = tbn_obs["nac"] * tbn_obs["year"].map(factor) / tbn_obs["poblacion"] * 1000
tbn_obs["tbn_suave_mm3"] = tbn_obs["tbn"].rolling(3, center=True, min_periods=1).mean()
tbn_obs.to_csv(DIR_TAB / "tbn_observada.csv", index=False)
tbn_obs.round(2)

,year,nac,poblacion,tbn,tbn_suave_mm3
0,2016,474185,39218146.0,13.22,12.33
1,2017,415385,39666414.0,11.45,12.20
2,2018,437350,40128882.0,11.92,11.58
3,2019,414514,39832799.0,11.38,11.09
4,2020,368227,40399992.0,9.97,9.99
5,2021,329077,41693498.0,8.63,9.63
6,2022,397600,42268056.0,10.28,8.94
7,2023,307157,42528063.0,7.90,8.98
8,2024,344175,42936914.0,8.76,8.33


## Descendencia final por generación

In [7]:
modelo_children = joblib.load(DIR_MOD / "modelo_children.joblib")
FEATURES_CHILDREN = joblib.load(DIR_MOD / "metadata_modelos.joblib")["features_children"]
ANIO_MAX_ENTRENAMIENTO = 2025
curva = panel_eph.groupby("age").apply(lambda g: np.average(g["target_children"], weights=g["pondera"]))
EDAD_REF = int(curva.idxmax())
print(f"edad de mayor acumulación de hijos convivientes: {EDAD_REF}")


def _features(df, edad, anio):
    X = df.copy()
    X["age"], X["age2"], X["year"] = float(edad), float(edad) ** 2, float(min(anio, ANIO_MAX_ENTRENAMIENTO))
    for c in FEATURES_CHILDREN:
        if c not in X.columns:
            X[c] = 0
    return X[FEATURES_CHILDREN]


def completar_descendencia(df, col_edad, col_anio):
    """Incremento hasta la edad de referencia, predicho fila a fila con el año fijo."""
    inc = np.zeros(len(df))
    for (e, a), idx in df.groupby([col_edad, col_anio]).indices.items():
        if int(e) == EDAD_REF:
            continue
        sub = df.iloc[idx]
        inc[idx] = np.maximum(modelo_children.predict(_features(sub, EDAD_REF, a))
                              - modelo_children.predict(_features(sub, e, a)), 0.0)
    return inc


ult = panel_eph.groupby("birth_year")["year"].max().rename("anio_obs").reset_index()
eph_ult = panel_eph.merge(ult, left_on=["birth_year", "year"], right_on=["birth_year", "anio_obs"], how="inner")
eph_ult["edad_obs"] = eph_ult["anio_obs"] - eph_ult["birth_year"]
eph_ult = eph_ult[eph_ult["edad_obs"].between(15, 50)].copy()
eph_ult["incremento_modelo"] = completar_descendencia(eph_ult, "edad_obs", "anio_obs")
eph_ult["descendencia_final"] = eph_ult["target_children"] + eph_ult["incremento_modelo"]
coh_eph = pd.DataFrame([{"cohorte": int(by), "fuente": "EPH", "n_mujeres": d["pondera"].sum(),
                         "edad_observacion": int(d["edad_obs"].iloc[0]),
                         "paridad_observada": np.average(d["target_children"], weights=d["pondera"]),
                         "incremento_modelo": np.average(d["incremento_modelo"], weights=d["pondera"]),
                         "descendencia_final": np.average(d["descendencia_final"], weights=d["pondera"])}
                        for by, d in eph_ult.groupby("birth_year") if d["pondera"].sum() > 0])
coh_eph["pct_extrapolado"] = 100 * coh_eph["incremento_modelo"] / coh_eph["descendencia_final"]
seguro_formalidad = panel_eph.groupby("education")[["insurance", "formality"]].mean()

GENERACIONES = [(1946, 1964, "Baby boom (1946-1964)"), (1965, 1980, "Generación X (1965-1980)"),
                (1981, 1996, "Millennials (1981-1996)"), (1997, 2012, "Centennials (1997-2012)"),
                (2013, 2025, "Generación Alfa (2013-2025)"), (2026, 2039, "Generación Beta (2026-2039)")]
UMBRAL_FILA_REPORTABLE = 30.0


def generacion(by):
    for lo, hi, et in GENERACIONES:
        if lo <= by <= hi:
            return et
    return None


def fecundidad_generacional(escenario):
    ult_sim = SIM[escenario]["cohortes"]
    coh_sim = (ult_sim.groupby("birth_year").agg(n_mujeres=("women_n", "median"), anio_observacion=("year", "median"),
                                                 edad_observacion=("edad", "median"),
                                                 paridad_observada=("paridad_media", "median"))
               .reset_index().rename(columns={"birth_year": "cohorte"}))
    comp_edu = SIM[escenario]["edad_educ"].groupby(["year", "age", "education"], as_index=False)["women_n"].median()
    comp_edu[["year", "age", "education"]] = comp_edu[["year", "age", "education"]].astype(int)
    incs = []
    for _, r in coh_sim.iterrows():
        edad, anio = int(r["edad_observacion"]), int(r["anio_observacion"])
        sub = comp_edu[(comp_edu.year == anio) & (comp_edu.age == edad)].copy()
        if edad == EDAD_REF:
            incs.append(0.0); continue
        if sub.empty or sub["women_n"].sum() <= 0:
            incs.append(np.nan); continue
        sub = sub.rename(columns={"women_n": "pondera"}).reset_index(drop=True)
        sub["region"], sub["couple"], sub["hh_work"] = 1, 1, 1
        imp = seguro_formalidad.reindex(sub["education"].astype(int)).reset_index(drop=True)
        sub["insurance"], sub["formality"] = imp["insurance"].values, imp["formality"].values
        for s_ in STATUS_LEVELS:
            sub[f"status{s_}"] = 1.0 / (len(STATUS_LEVELS) + 1)
        inc = completar_descendencia(sub.assign(_e=edad, _a=anio), "_e", "_a")
        incs.append(float(np.average(inc, weights=sub["pondera"])))
    coh_sim["incremento_modelo"] = incs
    coh_sim["descendencia_final"] = coh_sim["paridad_observada"] + coh_sim["incremento_modelo"].fillna(0)
    coh_sim["pct_extrapolado"] = (100 * coh_sim["incremento_modelo"].fillna(0)
                                  / coh_sim["descendencia_final"].replace(0, np.nan))
    coh_sim["fuente"] = "Simulación"

    cohortes = (pd.concat([coh_eph.assign(_pri=0), coh_sim.assign(_pri=1)], ignore_index=True)
                .sort_values(["cohorte", "_pri"]).drop_duplicates("cohorte", keep="first").drop(columns="_pri"))
    cohortes["generacion"] = cohortes["cohorte"].map(generacion)
    cohortes = cohortes.dropna(subset=["generacion", "descendencia_final"])
    filas = []
    for (gen, fuente), d in cohortes.groupby(["generacion", "fuente"]):
        w = d["n_mujeres"]
        filas.append({"generacion": gen, "fuente": fuente, "cohorte_min": int(d["cohorte"].min()),
                      "cohorte_max": int(d["cohorte"].max()),
                      "edad_media_obs": np.average(d["edad_observacion"], weights=w),
                      "paridad_observada": np.average(d["paridad_observada"], weights=w),
                      "incremento_modelo": np.average(d["incremento_modelo"].fillna(0), weights=w),
                      "descendencia_final": np.average(d["descendencia_final"], weights=w),
                      "pct_extrapolado": np.average(d["pct_extrapolado"].fillna(0), weights=w)})
    tabla = pd.DataFrame(filas)
    tabla["_orden"] = tabla["generacion"].map({et: i for i, (_, _, et) in enumerate(GENERACIONES)})
    tabla = tabla[tabla["pct_extrapolado"] < UMBRAL_FILA_REPORTABLE].sort_values("_orden").drop(columns="_orden")
    tabla["var_pct_vs_gen_mas_antigua"] = (tabla["descendencia_final"] / tabla["descendencia_final"].iloc[0] - 1) * 100
    tabla.to_csv(DIR_TAB / f"fecundidad_generacional_{escenario}.csv", index=False)
    return tabla.reset_index(drop=True)


generaciones = {e: fecundidad_generacional(e) for e in ESCENARIOS}
generaciones["base"][["generacion", "fuente", "descendencia_final"]].merge(
    generaciones["rebote"][["generacion", "descendencia_final"]], on="generacion", suffixes=("_base", "_rebote")).round(3)

edad de mayor acumulación de hijos convivientes: 42


,generacion,fuente,descendencia_final_base,descendencia_final_rebote
0,Generación X (1965-1980),EPH,1.895,1.895
1,Millennials (1981-1996),EPH,1.852,1.852
2,Centennials (1997-2012),Simulación,1.151,1.229
3,Generación Alfa (2013-2025),Simulación,1.121,1.253
4,Generación Beta (2026-2039),Simulación,0.986,1.206
